# AutoPolish — playground

*Refining images with frozen synthetic audiences.* &nbsp;·&nbsp; [paper](../docs/paper/neurips_creative_ai/autopolish.pdf) &nbsp;·&nbsp; [reproduction guide](../REPRODUCING.md)

A frozen vision-language model role-plays a **panel of viewers**. Their aggregated reaction
predicts how a real group responds to an image — and then drives an autonomous editing loop.
**No weights are ever trained.**

This notebook is a hands-on tour in four parts. Each part is independent — run only what you
have the hardware for.

| Part | What you get | Needs | Time |
|---|---|---|---|
| **1 · Meet the audience** | 10 personas react to one image; watch them disagree | GPU (~16 GB) | ~2 min |
| **2 · Why aggregation works** | the noise-cancelling effect, measured on your own panel | CPU | seconds |
| **3 · One AutoPolish step** | critic → instruction → edit → held-out judge, end to end | GPU (~24 GB) | ~3 min |
| **4 · The paper's numbers** | every headline figure, from cached results | CPU + data access | ~1 min |

Parts 1–3 use **only public models and two example images shipped in this repo** — no private
dataset access needed. Part 4 needs read access to the private `savoji/AUTOPOLISH` repo.


---
## 0 · Setup

Works locally or in Colab. In Colab this clones the repo; locally it just finds the root.


In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = os.environ.get("AUTOPOLISH_REPO", "")  # set to your clone URL
    if REPO_URL and not pathlib.Path("SyntheticAudience").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO = pathlib.Path("SyntheticAudience").resolve()
else:
    # notebook lives in <repo>/notebook/
    REPO = pathlib.Path.cwd()
    while not (REPO / "src").is_dir() and REPO != REPO.parent:
        REPO = REPO.parent

sys.path.insert(0, str(REPO / "src"))     # persona, editor
sys.path.insert(0, str(REPO / "script"))  # para_pipeline (prompt builders)
print("repo:", REPO)

# GPU check — Parts 1 and 3 need one; Parts 2 and 4 do not.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print("gpu:", torch.cuda.get_device_name(0) if HAS_GPU else "none (Parts 2 and 4 still work)")
except ImportError:
    HAS_GPU = False
    print("torch not installed — Parts 2 and 4 still work")

Install the inference stack **only if you plan to run Parts 1 or 3**. In Colab, restart the
runtime after this cell.


In [ ]:
# %pip install -q -r {REPO}/requirements.txt
# %pip install -q -r {REPO}/requirements-gpu.txt   # Parts 1 and 3 only

---
## 1 · Meet the synthetic audience &nbsp;<sub>GPU</sub>

Ten persona cards, one image, one frozen `Qwen2-VL-7B-Instruct`. Each persona is just **plain
text** — age, gender, education, art and photography experience, Big-Five scores — rendered
into a system prompt that asks the model to react *as that person*.

These ten cards use the exact format `script/para_pipeline.py` builds from real PARA
annotators, so what you see here is what the paper's panels saw.


In [ ]:
PANEL = [
    "a male aged 26-29, education level: senior high school; art experience: competent; "
    "photography experience: beginner. Big-Five personality scores (2-10 scale): "
    "openness 7/10 (high), conscientiousness 5/10 (medium), extraversion 6/10 (medium), "
    "agreeableness 7/10 (high), neuroticism 4/10 (low).",

    "a female aged 30-34, education level: university; art experience: proficient; "
    "photography experience: competent. Big-Five personality scores (2-10 scale): "
    "openness 9/10 (high), conscientiousness 7/10 (high), extraversion 4/10 (low), "
    "agreeableness 6/10 (medium), neuroticism 5/10 (medium).",

    "a male aged 35-40, education level: senior high school; art experience: expert; "
    "photography experience: proficient. Big-Five personality scores (2-10 scale): "
    "openness 8/10 (high), conscientiousness 6/10 (medium), extraversion 3/10 (low), "
    "agreeableness 5/10 (medium), neuroticism 6/10 (medium).",

    "a female aged 18-21, education level: junior college; art experience: beginner; "
    "photography experience: beginner. Big-Five personality scores (2-10 scale): "
    "openness 5/10 (medium), conscientiousness 4/10 (low), extraversion 8/10 (high), "
    "agreeableness 8/10 (high), neuroticism 5/10 (medium).",

    "a male aged 22-25, education level: university; art experience: competent; "
    "photography experience: expert. Big-Five personality scores (2-10 scale): "
    "openness 8/10 (high), conscientiousness 8/10 (high), extraversion 5/10 (medium), "
    "agreeableness 4/10 (low), neuroticism 3/10 (low).",

    "a female aged 41-50, education level: postgraduate; art experience: expert; "
    "photography experience: competent. Big-Five personality scores (2-10 scale): "
    "openness 9/10 (high), conscientiousness 7/10 (high), extraversion 4/10 (low), "
    "agreeableness 7/10 (high), neuroticism 4/10 (low).",

    "a male aged 30-34, education level: junior college; art experience: beginner; "
    "photography experience: beginner. Big-Five personality scores (2-10 scale): "
    "openness 4/10 (low), conscientiousness 6/10 (medium), extraversion 7/10 (high), "
    "agreeableness 6/10 (medium), neuroticism 7/10 (high).",

    "a female aged 26-29, education level: university; art experience: proficient; "
    "photography experience: beginner. Big-Five personality scores (2-10 scale): "
    "openness 7/10 (high), conscientiousness 5/10 (medium), extraversion 6/10 (medium), "
    "agreeableness 8/10 (high), neuroticism 6/10 (medium).",

    "a male aged 51-60, education level: senior high school; art experience: competent; "
    "photography experience: proficient. Big-Five personality scores (2-10 scale): "
    "openness 6/10 (medium), conscientiousness 9/10 (high), extraversion 3/10 (low), "
    "agreeableness 5/10 (medium), neuroticism 3/10 (low).",

    "a female aged 22-25, education level: postgraduate; art experience: expert; "
    "photography experience: expert. Big-Five personality scores (2-10 scale): "
    "openness 10/10 (high), conscientiousness 6/10 (medium), extraversion 5/10 (medium), "
    "agreeableness 4/10 (low), neuroticism 5/10 (medium).",
]
print(f"{len(PANEL)} persona cards")

Two example images ship with the repo — the same ones used in the paper's supplement.
**Swap in your own** by pointing `IMAGE` at any file or URL.


In [ ]:
from PIL import Image

EXAMPLES = {
    "office": REPO / "docs/paper/neurips_creative_ai/figs/ex_office_src.jpg",   # open-plan office, aesthetic 5.57
    "basketball": REPO / "docs/paper/neurips_creative_ai/figs/ex_ball_src.jpg",  # basketball on a dark floor
}
IMAGE = EXAMPLES["office"]      # <-- try "basketball", or any path/URL of your own

img = Image.open(IMAGE).convert("RGB")
print(IMAGE.name, img.size)
img.resize((img.width // 2, img.height // 2))

Load the frozen judge **once** and share it across all ten personas — that sharing is what
makes a panel cheap. First run downloads ~16 GB of weights.


In [ ]:
from persona import QwenVLBackend

backend = QwenVLBackend("Qwen/Qwen2-VL-7B-Instruct")   # frozen: never trained, never fine-tuned
print("judge ready")

Now run the panel. `SocietyCritic` is the paper's method: it batches one call per persona and
returns each viewer's score **and** the single change they would most want.


In [ ]:
import pandas as pd
import para_pipeline as para          # the prompt/parsing machinery the paper's runs used
from editor import SocietyCritic

critic = SocietyCritic(backend, PANEL)
critique = critic.critique(img)

# Re-parse each persona's raw reply so score and complaint stay attached to the
# viewer who said them (critique.panel_scores/.complaints drop blanks independently).
SCORE_DIM = para.ScoreDimension("score", 0.0, 100.0, 1.0, "overall appeal")
import re
def label(card):
    who = card.split(",")[0].removeprefix("a ")
    art = re.search(r"art experience: (\w+)", card)
    return f"{who}, art: {art.group(1)}" if art else who

rows = []
for card, raw in zip(PANEL, critique.raw):
    parsed, comment = para.parse_para_rating(raw, [SCORE_DIM])
    rows.append({"viewer": label(card),
                 "score": parsed.get("score"),
                 "requested change": comment or ""})

panel_df = pd.DataFrame(rows)
panel_df.style.hide(axis="index")

### The group reaction

The panel does **not** agree — and that is the point. The prediction is not any one viewer's
score but the *shape* of the whole reaction: its center, its spread, and the complaints that
recur across viewers.


In [ ]:
import numpy as np
from collections import Counter

s = panel_df["score"].dropna().to_numpy(dtype=float)
print(f"group mean   {s.mean():6.1f}  / 100")
print(f"spread (sd)  {s.std(ddof=1):6.1f}")
print(f"range        {s.min():.0f} - {s.max():.0f}")
print(f"n viewers    {len(s):6d}")
print("\nrecurring complaints:")
changes = [x.lower().rstrip('.') for x in panel_df["requested change"] if x]
for c, n in Counter(changes).most_common(5):
    print(f"  {n}x  {c}")

---
## 2 · Why aggregation works &nbsp;<sub>CPU</sub>

The paper's organizing measurement: **an individual rating is near-noise, but the group mean is
reliable.** On real human data a single rating shares only 19–47% of its variance with other
raters (ICC(1) 0.470 / 0.223 / 0.188 on PARA / EVA / LAPIS), while the *group mean* of ~25
raters reaches ICC(k) 0.84–0.96.

The mechanism is just noise cancellation: the idiosyncratic part of each reaction averages out.
You can watch it happen on the panel you just ran — the running mean settles as viewers are
added, while any single viewer stays scattered.


In [ ]:
import numpy as np, matplotlib.pyplot as plt

# Falls back to the paper's own office panel if you skipped Part 1.
scores = np.array(globals().get("s", [60, 60, 60, 60, 55, 65, 60, 70, 50, 60]), dtype=float)

# Draw panels of size N *with replacement* from the viewer population, 2000 times each.
# (Without replacement the full-panel mean is deterministic, which would hide the very
#  effect we are measuring.)
rng = np.random.default_rng(0)
MAXN, REPS = 20, 2000
draws = rng.choice(scores, size=(REPS, MAXN), replace=True)
running = np.cumsum(draws, axis=1) / np.arange(1, MAXN + 1)
ns = np.arange(1, MAXN + 1)
lo, hi = np.percentile(running, [2.5, 97.5], axis=0)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.fill_between(ns, lo, hi, alpha=0.15, color="#2a78d6", label="95% of resampled panels")
ax.plot(ns, running.mean(0), "-o", ms=4, color="#2a78d6", label="panel mean")
ax.axhline(scores.mean(), ls="--", lw=1, color="#8a8a86", label="population value")
ax.set_xlabel("panel size $N$"); ax.set_ylabel("predicted group score")
ax.set_title("The estimate stabilizes as the panel grows", fontsize=10)
ax.legend(fontsize=8, frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for n in (1, 5, 10, 20):
    print(f"panel of {n:2d}:  sd of the estimate = {running[:, n-1].std(ddof=1):5.2f}")
print(f"\nthe 1/sqrt(N) law: {scores.std(ddof=1):.2f} / sqrt(20) = "
      f"{scores.std(ddof=1) / np.sqrt(20):.2f}")

> **The honest caveat.** On *real* images this curve is flatter than it should be, because
> greedy decoding collapses the panel toward one value (the backbone reverts to the mode ~88% of
> the time on discrete rating tasks). The effect shows cleanly on AI-generated image pairs
> instead, where accuracy climbs monotonically from 0.588 at N=1 to 0.657 at N=20. Paper §6,
> limitation 1 — try `do_sample=True, temperature=0.7` above to see the spread widen.


---
## 3 · One AutoPolish step &nbsp;<sub>GPU</sub>

Now put the audience to work. One full turn of the loop:

```
panel complaints ──> distill to ONE ≤15-word instruction ──> FLUX edit ──┐
                                                                        ▼
        commit only if BOTH improve <── held-out judge (LAION aesthetic + DINOv2 identity)
```

The critic that proposes and the judge that grades are **different model families**, and the
judge never sees the instruction. That is what stops the loop from grading its own homework.


In [ ]:
from editor import distill_instruction

instruction = distill_instruction(backend, img, accumulated="", new_complaints=critique.complaints)
print("panel complaints:")
for c in critique.complaints[:5]:
    print("  -", c)
print(f"\ndistilled instruction:\n  \u201c{instruction}\u201d")

In [ ]:
from editor import build_editor

editor = build_editor(
    "flux",
    model_name="black-forest-labs/FLUX.1-Kontext-dev",  # gated repo: accept its license first
    cpu_offload=True,   # streams weights; set False on an A100/H100 for speed
)
candidates = editor.edit(img, instruction, k=2, seed=0)
print(f"{len(candidates)} candidate edit(s)")

### The held-out judge decides

Two independent checks, neither of which the critic can see: a LAION aesthetic predictor (CLIP
backbone + MLP head) and a DINOv2 identity similarity that must stay above **0.78** so an
"improvement" cannot just be a different picture.


In [ ]:
from editor import AestheticObjective, DriftMetric

objective = AestheticObjective(device="cuda")
drift = DriftMetric(device="cuda")
DRIFT_CAP = 0.78

base = objective.score(img)
rows = []
for i, cand in enumerate(candidates):
    aes, ident = objective.score(cand), drift.similarity(img, cand)
    rows.append({"candidate": i, "aesthetic": round(aes, 3), "gain": round(aes - base, 3),
                 "identity": round(ident, 3),
                 "committed": aes > base and ident >= DRIFT_CAP})

import pandas as pd
print(f"source aesthetic: {base:.3f}\n")
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
# side by side: source vs the best committed candidate
import matplotlib.pyplot as plt

ok = [r for r in rows if r["committed"]]
best = candidates[max(ok, key=lambda r: r["aesthetic"])["candidate"]] if ok else None

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
axes[0].imshow(img); axes[0].set_title(f"source — {base:.2f}", fontsize=10)
if best is not None:
    b = max(ok, key=lambda r: r["aesthetic"])
    axes[1].imshow(best)
    axes[1].set_title(f"committed — {b['aesthetic']:.2f}  (identity {b['identity']:.2f})",
                      fontsize=10, color="#2a78d6")
else:
    axes[1].text(.5, .5, "nothing committed\n(no candidate improved\nwhile preserving identity)",
                 ha="center", va="center", fontsize=10)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

**Rejection is a feature.** If nothing beat the source while preserving identity, the loop keeps
the original and tries again next step with accumulated complaints. Over 10 steps the best-so-far
trajectory is monotone *by construction* — which is exactly why the objective has to be held out.

The full 10-step, 4-condition experiment is `scripts/run_c4.sh` (see
[REPRODUCING.md](../REPRODUCING.md) tier 3).


---
## 4 · The paper's headline numbers &nbsp;<sub>CPU · needs data access</sub>

Everything in the paper is re-derivable from cached analysis JSONs. This pulls them and prints
the headline table.

Needs read access to the private `savoji/AUTOPOLISH` repo and `HF_TOKEN` in `.env`
(~2.2 GB download, ~4.5 GB free while it runs). If you don't have access, skip this part —
the numbers are all in [the paper](../docs/paper/neurips_creative_ai/autopolish.pdf).


In [ ]:
# !cd {REPO} && python scripts/fetch_from_hf.py autopolish

In [ ]:
import json, pathlib

RES = REPO / "results"
if not RES.exists():
    print("results/ not found — run the fetch cell above (needs access to savoji/AUTOPOLISH)")
else:
    load = lambda n: json.load(open(RES / f"{n}.json"))
    exp0, c1, c3 = load("exp0"), load("c1_separation"), load("c3")

    print("THE CEILING  (paper Table S1)")
    print(f"  {'dataset':8} {'ICC(1)':>8} {'ICC(k)':>8}   one rating vs the group mean")
    for d in ["PARA", "EVA", "LAPIS"]:
        v = exp0[d]["variance_decomposition"]
        print(f"  {d:8} {v['ICC1_single_rating']:8.3f} {v['ICCk_group_mean']:8.3f}")

    print("\nBETWEEN-GROUP SEPARATION  (paper Fig. 2b)")
    print(f"  {'dataset':8} {'panel':>8} {'blind':>8}   95% CI")
    for d in ["PARA", "EVA", "LAPIS"]:
        o = c1[d]["overall"]
        lo, hi = o["full_separation"]["ci95"]
        print(f"  {d:8} {o['full_separation']['corr']:+8.3f} "
              f"{o['blind_separation']['corr']:+8.3f}   [{lo:+.3f}, {hi:+.3f}]")

    print("\nGENERATED IMAGES  (paper Fig. 2c)")
    a = c3["aggregation"]
    print(f"  panel-vs-crowd correlation  {a['pair_corr']:+.3f}")
    print(f"  panel accuracy              {a['aggregate_acc']:.3f}  vs majority prior "
          f"{a['aggregate_acc_majority']:.3f}")
    print("  panel size ->  " + "  ".join(f"N={k}: {v:.3f}" for k, v in a["n_curve"].items()))

To rebuild the paper's figures from these same JSONs:

```bash
cd scripts/analysis
python paper_figs.py --c4-root ../../data/results/c4_run2
cd ../../docs/paper/neurips_creative_ai && latexmk -pdf autopolish.tex
```


---
## Where to go next

| | |
|---|---|
| **Read the result** | [`docs/paper/neurips_creative_ai/autopolish.pdf`](../docs/paper/neurips_creative_ai/autopolish.pdf) |
| **Reproduce it** | [`REPRODUCING.md`](../REPRODUCING.md) — three tiers, cheapest first |
| **Understand the code** | [`docs/architecture.md`](../docs/architecture.md) |
| **Run the full experiment** | `scripts/run_c4.sh` · [`docs/RUNNING_ON_H100.md`](../docs/RUNNING_ON_H100.md) |
| **See what was tried** | [`research_plan.md`](../research_plan.md) §14 — the full interim log |

### Things worth trying in this notebook

- **Change the panel.** Edit `PANEL` to a different demographic and watch the group mean move.
  Persona signal tracks how richly viewers are described — that ordering (LAPIS > PARA > EVA) is
  the paper's cleanest evidence that the persona, not a confound, is doing the work.
- **Compare critics.** Swap `SocietyCritic` for `BlindVLMCritic(backend)` in Part 3 and compare
  the distilled instructions. The society critic produces 3.1× more distinct complaints.
- **Turn off the persona.** Feed the same image through `build_critic("blind", backend)` — this
  is the control that scores ~0 on between-group separation.
- **Raise the temperature.** `do_sample=True, temperature=0.7` widens the panel and is the
  workaround for the greedy-decoding collapse discussed in Part 2.
